# Project 2 — Deep-Dive EDA

<small>Annotation: this notebook is the exploratory audit trail behind the executive figures and modeling rationale.</small>

*Ported from Gloria's `Project 2` notebook on `main`, adapted to the `bookrec` package.*

This is a **richer companion** to the condensed EDA in `01_build_book_recommender.ipynb`
(Step 2). It loads the data through the package's `data_loader` — the same
`[user_id, book_id, rating]` contract used everywhere else in the project — and then
explores user behavior, book behavior, popularity concentration, and metadata quality
in more depth. These are the angles most useful for *justifying* the modeling choices
made in notebook 01 (popularity baseline, UBCF/IBCF, SVD).

In [ ]:
# Notebook-local setup: expose ../src so the EDA uses the same loader as the app.
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # import the bookrec `src` package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data_loader

In [ ]:
# Load via the package loader: repairs author/title mojibake and enforces the
# [user_id, book_id, rating] ratings contract. `books` keeps its raw metadata
# columns (isbn, language_code, text_reviews_count, ...) plus the contract
# defaults the loader fills (e.g. missing authors -> "Unknown", year -> 0).
ratings, books = data_loader.load('../data')

# Display lookups used below.
title_of = dict(zip(books['book_id'], books['title']))
label_of = {
    row.book_id: f"{row.title} — {row.authors}"
    for row in books[['book_id', 'title', 'authors']].itertuples(index=False)
}

print(f"Books: {books.shape}  |  Ratings: {ratings.shape}")
display(books.head())
display(ratings.head())

## Data quality

Sanity checks after the loader's cleanup — duplicates, orphan ratings, and missing keys.
A clean table here is what lets us trust every downstream metric.

In [ ]:
# Build a compact audit table instead of checking each issue manually in prose.
quality_checks = pd.DataFrame({
    "check": [
        "Duplicate book IDs",
        "Duplicate user-book ratings",
        "Ratings with no matching book",
        "Missing book titles",
        "Missing user IDs",
        "Missing book IDs in ratings",
        "Missing ratings"
    ],
    "count": [
        books["book_id"].duplicated().sum(),
        ratings.duplicated(["user_id", "book_id"]).sum(),
        (~ratings["book_id"].isin(books["book_id"])).sum(),
        books["title"].isna().sum(),
        ratings["user_id"].isna().sum(),
        ratings["book_id"].isna().sum(),
        ratings["rating"].isna().sum()
    ]
})
display(quality_checks)

## Rating distribution

Recommendation data is usually skewed toward high ratings (people rate books they
already expected to like — *missing-not-at-random*). A heavy 4–5★ skew is the single
biggest reason a popularity/mean baseline is hard to beat on RMSE.

In [ ]:
# Count and percent together make the rating skew visible at a glance.
rating_dist = (
    ratings["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)
rating_dist["percent"] = (rating_dist["count"] / len(ratings) * 100).round(2)
display(rating_dist)

print("Mean rating:", round(ratings["rating"].mean(), 3))
print("Median rating:", ratings["rating"].median())

plt.figure(figsize=(7, 4))
plt.bar(rating_dist["rating"], rating_dist["count"], color="steelblue", edgecolor="white")
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.xticks(rating_dist["rating"])
plt.tight_layout()
plt.show()

## User behavior

How much history each user has, and how generous or harsh they are. Users with very
short histories or a single rating value are exactly the cold-start / low-signal cases
where collaborative filtering struggles and a popularity fallback matters.

In [ ]:
user_summary = (
    ratings.groupby("user_id")
    .agg(
        n_ratings=("rating", "count"),
        avg_rating=("rating", "mean"),
        rating_std=("rating", "std"),
        min_rating=("rating", "min"),
        max_rating=("rating", "max"),
        unique_ratings=("rating", "nunique")
    )
    .reset_index()
)
display(user_summary.describe())

single_value_users = user_summary[user_summary["unique_ratings"] == 1]
all_five_users = single_value_users[single_value_users["avg_rating"] == 5]
print("Users who only use one rating value:", len(single_value_users))
print("Users who only gave 5-star ratings:", len(all_five_users))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(user_summary["n_ratings"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Ratings Per User")
axes[0].set_xlabel("Number of Ratings")
axes[0].set_ylabel("Users")
axes[1].hist(user_summary["avg_rating"], bins=30, color="steelblue", edgecolor="white")
axes[1].set_title("Average Rating Per User")
axes[1].set_xlabel("Average Rating")
plt.tight_layout()
plt.show()

## Book behavior

How much evidence each book carries, and how its observed average relates to its rating
count. The scatter shows whether high averages are backed by enough ratings to trust —
the motivation for the support threshold below.

In [ ]:
book_summary = (
    ratings.groupby("book_id")
    .agg(
        observed_ratings=("rating", "count"),
        observed_avg_rating=("rating", "mean"),
        observed_rating_std=("rating", "std")
    )
    .reset_index()
    .merge(
        books[[
            "book_id", "title", "authors", "original_publication_year",
            "language_code", "average_rating", "ratings_count", "text_reviews_count"
        ]],
        on="book_id", how="left"
    )
)
display(book_summary.describe())

print("Most-rated books in this sample:")
display(
    book_summary.sort_values("observed_ratings", ascending=False).head(15)
    [["book_id", "title", "authors", "observed_ratings", "observed_avg_rating"]]
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(book_summary["observed_ratings"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Ratings Per Book")
axes[0].set_xlabel("Observed Ratings")
axes[0].set_ylabel("Books")
axes[1].scatter(book_summary["observed_ratings"], book_summary["observed_avg_rating"], alpha=0.35)
axes[1].set_title("Book Rating Count vs Average Rating")
axes[1].set_xlabel("Observed Ratings")
axes[1].set_ylabel("Observed Average Rating")
plt.tight_layout()
plt.show()

In [ ]:
# Top and bottom books, using a support threshold so single-rating books
# do not dominate the leaderboard.
MIN_RATINGS = 20
supported_books = book_summary[book_summary["observed_ratings"] >= MIN_RATINGS]

print(f"Highest-rated books with at least {MIN_RATINGS} ratings:")
display(
    supported_books.sort_values("observed_avg_rating", ascending=False).head(10)
    [["title", "authors", "observed_ratings", "observed_avg_rating", "average_rating"]]
)
print(f"Lowest-rated books with at least {MIN_RATINGS} ratings:")
display(
    supported_books.sort_values("observed_avg_rating", ascending=True).head(10)
    [["title", "authors", "observed_ratings", "observed_avg_rating", "average_rating"]]
)

## Popularity concentration

Does a small set of books soak up most of the ratings? A steep long tail means a naive
recommender will keep surfacing the same blockbusters — the argument for measuring
**coverage** (notebook 01, `evaluate.coverage`) alongside accuracy.

In [ ]:
book_popularity = ratings["book_id"].value_counts()

popularity_summary = pd.DataFrame({"top_n_books": [10, 50, 100, 500, 1000]})
popularity_summary["percent_of_all_ratings"] = popularity_summary["top_n_books"].apply(
    lambda n: round(100 * book_popularity.head(n).sum() / len(ratings), 2)
)
display(popularity_summary)

plt.figure(figsize=(7, 4))
book_popularity.reset_index(drop=True).plot(color="steelblue")
plt.title("Book Popularity Long Tail")
plt.xlabel("Books Ranked by Popularity")
plt.ylabel("Number of Ratings")
plt.tight_layout()
plt.show()

## Metadata & missingness

Coverage of the side information a content-based or hybrid model would rely on.

> **Note:** `data_loader.load` fills a few contract columns (missing `authors` → `"Unknown"`,
> missing `original_publication_year` → `0`), so those missing-counts read as 0 here by
> design — they reflect the **post-clean** state the models actually see. `language_code`
> and `isbn` are left untouched, so their gaps are real.

In [ ]:
metadata_summary = pd.DataFrame({
    "column": ["language_code", "isbn", "original_publication_year", "authors"],
    "missing_count": [
        books["language_code"].isna().sum(),
        books["isbn"].isna().sum(),
        books["original_publication_year"].isna().sum(),
        books["authors"].isna().sum()
    ]
})
metadata_summary["missing_percent"] = (
    metadata_summary["missing_count"] / len(books) * 100
).round(2)
display(metadata_summary)

print("Most common languages:")
display(books["language_code"].fillna("Missing").value_counts().head(15).to_frame("book_count"))

print("Most common authors:")
display(
    books["authors"].dropna().str.split(",").explode().str.strip()
    .value_counts().head(20).to_frame("book_count")
)

print("Potential publication year anomalies:")
display(
    books[(books["original_publication_year"] < 0) | (books["original_publication_year"] > 2026)]
    [["book_id", "title", "authors", "original_publication_year"]].head(20)
)

## EDA takeaways

- **Skewed ratings (heavy 4–5★).** Rating data is missing-not-at-random, so RMSE
  rewards predicting "high" — a mean/popularity baseline is a genuinely strong
  competitor and the right thing to benchmark against (notebook 01, Step 3).
- **Sparse, uneven users.** Many users have short histories or near-constant ratings →
  cold-start cases where CF neighborhoods are thin; keep a popularity fallback.
- **Long-tail popularity.** How steep the tail is varies — read it off the concentration
  table above. The steeper it is, the more you should report **coverage** next to
  Precision/Recall/NDCG rather than accuracy alone.
- **Partial metadata.** `language_code`/`isbn` gaps and author concentration constrain a
  content/hybrid model — the content path falls back to title + authors for a reason.